# Notebook 2 — Feature Engineering

**Input:** Raw JSON + CSV files from `data/train/` and `data/test/`

**Output:** `features_train.csv` and `features_test.csv` (one row per applicant, all features + TARGET)

In [1]:
import json
import numpy as np
import pandas as pd

# ============================================================
# PATHS
# ============================================================
TRAIN_FLAG   = r'E:\senior_ds_test\senior_ds_test\data\raw\train_flag.csv'
TRAIN_ACC    = r'E:\senior_ds_test\senior_ds_test\data\raw\accounts_data_train.json'
TRAIN_ENQ    = r'E:\senior_ds_test\senior_ds_test\data\raw\enquiry_data_train.json'
TEST_ACC     = r'E:\senior_ds_test\senior_ds_test\data\raw\accounts_data_test.json'
TEST_ENQ     = r'E:\senior_ds_test\senior_ds_test\data\raw\enquiry_data_test.json'
TEST_FLAG  = r'E:\senior_ds_test\senior_ds_test\data\raw\test_flag.csv'

OUT_TRAIN  = 'features_train.csv'
OUT_TEST   = 'features_test.csv'

REF_DATE = pd.Timestamp('2021-01-01')  # confirmed snapshot date from EDA

In [2]:
def load_nested_json(path):
    """Flatten nested list-of-lists JSON into a tidy DataFrame."""
    with open(path) as f:
        raw = json.load(f)
    records = [rec for applicant in raw for rec in applicant if rec is not None]
    return pd.DataFrame(records)

## 1. Account Features

In [3]:
KEEP_CREDIT_TYPES = ['Consumer credit', 'Credit card', 'Car loan', 'Mortgage', 'Microloan']

def build_account_features(acc):
    acc = acc.copy()

    # dtypes
    acc['open_date']      = pd.to_datetime(acc['open_date'], errors='coerce')
    acc['closed_date']    = pd.to_datetime(acc['closed_date'], errors='coerce')
    acc['loan_amount']    = pd.to_numeric(acc['loan_amount'], errors='coerce')
    acc['amount_overdue'] = pd.to_numeric(acc['amount_overdue'], errors='coerce')

    # fix garbage closed_dates
    bad = acc['closed_date'] < acc['open_date']
    acc.loc[bad, 'closed_date'] = pd.NaT

    # derived per-row fields
    acc['is_open']       = acc['closed_date'].isna().astype(int)
    acc['is_zero_amt']   = (acc['loan_amount'] == 0).astype(int)
    acc['has_overdue']   = (acc['amount_overdue'] > 0).astype(int)
    acc['days_since_open']  = (REF_DATE - acc['open_date']).dt.days
    acc['days_since_close'] = (REF_DATE - acc['closed_date']).dt.days
    acc['log_loan_amount']  = np.log1p(acc['loan_amount'].clip(lower=0))
    acc['credit_grp'] = np.where(acc['credit_type'].isin(KEEP_CREDIT_TYPES),
                                 acc['credit_type'], 'Other')

    g = acc.groupby('uid')

    num = g.agg(
        acc_n_accounts       = ('uid', 'size'),
        acc_n_open           = ('is_open', 'sum'),
        acc_n_zero_amt       = ('is_zero_amt', 'sum'),
        acc_loan_amt_sum     = ('loan_amount', 'sum'),
        acc_loan_amt_mean    = ('loan_amount', 'mean'),
        acc_loan_amt_max     = ('loan_amount', 'max'),
        acc_loan_amt_std     = ('loan_amount', 'std'),
        acc_log_loan_sum     = ('log_loan_amount', 'sum'),
        acc_log_loan_mean    = ('log_loan_amount', 'mean'),
        acc_overdue_sum      = ('amount_overdue', 'sum'),
        acc_overdue_max      = ('amount_overdue', 'max'),
        acc_n_with_overdue   = ('has_overdue', 'sum'),
        acc_n_credit_types   = ('credit_type', 'nunique'),
        acc_days_since_open_min  = ('days_since_open', 'min'),
        acc_days_since_open_max  = ('days_since_open', 'max'),
        acc_days_since_open_mean = ('days_since_open', 'mean'),
        acc_days_since_close_min = ('days_since_close', 'min'),
    )

    num['acc_open_ratio']        = num['acc_n_open']         / num['acc_n_accounts']
    num['acc_overdue_ratio']     = num['acc_n_with_overdue'] / num['acc_n_accounts']
    num['acc_overdue_amt_ratio'] = num['acc_overdue_sum']    / (num['acc_loan_amt_sum'] + 1)
    num['acc_zero_amt_ratio']    = num['acc_n_zero_amt']     / num['acc_n_accounts']

    cat = (acc.groupby(['uid', 'credit_grp']).size()
              .unstack(fill_value=0).add_prefix('acc_cnt_'))
    cat_share = cat.div(cat.sum(axis=1), axis=0).add_suffix('_share')

    out = num.join(cat).join(cat_share)
    return out.reset_index()

print('build_account_features defined ✓')

build_account_features defined ✓


## 2. Payment History Features

In [4]:
def _parse_ph(s):
    """Return list of monthly DPD ints; most recent is last."""
    s = str(s)
    if s in ('', 'nan') or len(s) < 3:
        return []
    n = len(s) - (len(s) % 3)
    return [int(s[i:i+3]) for i in range(0, n, 3)]

def _ph_row_features(s):
    vals = _parse_ph(s)
    if not vals:
        return dict(ph_n_months=0, ph_max_dpd=0, ph_mean_dpd=0, ph_sum_dpd=0,
                    ph_n_late=0, ph_n_30=0, ph_n_90=0, ph_n_180=0, ph_n_360=0,
                    ph_recent_dpd=0, ph_recent3_max=0, ph_late_ratio=0,
                    ph_has_history=0)
    arr = np.array(vals)
    return dict(
        ph_n_months   = len(arr),
        ph_max_dpd    = int(arr.max()),
        ph_mean_dpd   = float(arr.mean()),
        ph_sum_dpd    = int(arr.sum()),
        ph_n_late     = int((arr > 0).sum()),
        ph_n_30       = int((arr >= 30).sum()),
        ph_n_90       = int((arr >= 90).sum()),
        ph_n_180      = int((arr >= 180).sum()),
        ph_n_360      = int((arr >= 360).sum()),
        ph_recent_dpd = int(arr[-1]),
        ph_recent3_max= int(arr[-3:].max()),
        ph_late_ratio = float((arr > 0).mean()),
        ph_has_history= 1,
    )

def build_payment_features(acc):
    acc = acc.copy()
    ph = acc['payment_hist_string'].apply(_ph_row_features).apply(pd.Series)
    ph['uid'] = acc['uid'].values

    g = ph.groupby('uid')
    out = g.agg(
        pmt_max_dpd        = ('ph_max_dpd', 'max'),
        pmt_mean_dpd       = ('ph_mean_dpd', 'mean'),
        pmt_sum_dpd        = ('ph_sum_dpd', 'sum'),
        pmt_n_late         = ('ph_n_late', 'sum'),
        pmt_n_30           = ('ph_n_30', 'sum'),
        pmt_n_90           = ('ph_n_90', 'sum'),
        pmt_n_180          = ('ph_n_180', 'sum'),
        pmt_n_360          = ('ph_n_360', 'sum'),
        pmt_recent_dpd_max = ('ph_recent_dpd', 'max'),
        pmt_recent3_max    = ('ph_recent3_max', 'max'),
        pmt_total_months   = ('ph_n_months', 'sum'),
        pmt_late_ratio_max = ('ph_late_ratio', 'max'),
        pmt_late_ratio_mean= ('ph_late_ratio', 'mean'),
        pmt_n_acc_with_hist= ('ph_has_history', 'sum'),
    )

    out['pmt_ever_90plus']  = (out['pmt_n_90']  > 0).astype(int)
    out['pmt_ever_180plus'] = (out['pmt_n_180'] > 0).astype(int)
    out['pmt_ever_360plus'] = (out['pmt_n_360'] > 0).astype(int)

    return out.reset_index()

print('build_payment_features defined ✓')

build_payment_features defined ✓


## 3. Enquiry Features

In [5]:
KEEP_ENQ_TYPES = ['Cash loans', 'Revolving loans']

def build_enquiry_features(enq):
    enq = enq.copy()
    enq['enquiry_date'] = pd.to_datetime(enq['enquiry_date'], errors='coerce')
    enq['enquiry_amt']  = pd.to_numeric(enq['enquiry_amt'], errors='coerce')
    enq['log_enq_amt']  = np.log1p(enq['enquiry_amt'].clip(lower=0))
    enq['days_ago']     = (REF_DATE - enq['enquiry_date']).dt.days

    for w in [30, 90, 180, 365]:
        enq[f'in_{w}d'] = (enq['days_ago'] <= w).astype(int)

    g = enq.groupby('uid')
    num = g.agg(
        enq_n              = ('uid', 'size'),
        enq_amt_sum        = ('enquiry_amt', 'sum'),
        enq_amt_mean       = ('enquiry_amt', 'mean'),
        enq_amt_max        = ('enquiry_amt', 'max'),
        enq_log_amt_sum    = ('log_enq_amt', 'sum'),
        enq_log_amt_mean   = ('log_enq_amt', 'mean'),
        enq_n_types        = ('enquiry_type', 'nunique'),
        enq_days_since_last  = ('days_ago', 'min'),
        enq_days_since_first = ('days_ago', 'max'),
        enq_n_30d          = ('in_30d', 'sum'),
        enq_n_90d          = ('in_90d', 'sum'),
        enq_n_180d         = ('in_180d', 'sum'),
        enq_n_365d         = ('in_365d', 'sum'),
    )
    num['enq_span_days'] = num['enq_days_since_first'] - num['enq_days_since_last']

    enq['enq_grp'] = np.where(enq['enquiry_type'].isin(KEEP_ENQ_TYPES),
                              enq['enquiry_type'], 'Other')
    cat = (enq.groupby(['uid', 'enq_grp']).size()
              .unstack(fill_value=0).add_prefix('enq_cnt_'))

    out = num.join(cat)
    return out.reset_index()

print('build_enquiry_features defined ✓')

build_enquiry_features defined ✓


## 4. Master Builder

In [6]:
def build_master(flag, acc, enq):
    m = flag.copy()
    m['is_cash_loan'] = (m['NAME_CONTRACT_TYPE'] == 'Cash loans').astype(int)

    m = m.merge(build_account_features(acc), on='uid', how='left')
    m = m.merge(build_payment_features(acc), on='uid', how='left')
    m = m.merge(build_enquiry_features(enq), on='uid', how='left')

    m['has_accounts'] = m['acc_n_accounts'].notna().astype(int)

    acc_cols = [c for c in m.columns if c.startswith(('acc_', 'pmt_'))]
    m[acc_cols] = m[acc_cols].fillna(0)
    enq_cols = [c for c in m.columns if c.startswith('enq_')]
    m[enq_cols] = m[enq_cols].fillna(0)
    m = m.fillna(0)

    return m

print('build_master defined ✓')

build_master defined ✓


## 5. Build & Save Features

In [7]:
# TRAIN
flag_train = pd.read_csv(TRAIN_FLAG)
acc_train  = load_nested_json(TRAIN_ACC)
enq_train  = load_nested_json(TRAIN_ENQ)
train = build_master(flag_train, acc_train, enq_train)

# TEST
flag_test = pd.read_csv(TEST_FLAG)
acc_test  = load_nested_json(TEST_ACC)
enq_test  = load_nested_json(TEST_ENQ)
test = build_master(flag_test, acc_test, enq_test)

print('Train:', train.shape)
print('Test :', test.shape)

Train: (261383, 72)
Test : (46127, 71)


In [11]:
# Align columns between train & test
feature_cols = [c for c in train.columns
                if c not in ['uid', 'TARGET', 'NAME_CONTRACT_TYPE']]

for c in feature_cols:
    if c not in test.columns:
        test[c] = 0

test = test[['uid'] + feature_cols]

print('Final feature count:', len(feature_cols))
print('Any NaN in train?', train[feature_cols].isna().any().any())
print('Any NaN in test? ', test[feature_cols].isna().any().any())
assert 'TARGET' not in feature_cols
assert train['uid'].is_unique
assert test['uid'].is_unique

Final feature count: 69
Any NaN in train? False
Any NaN in test?  False


In [9]:
# Quick sanity: do features separate the classes?
check_cols = ['pmt_max_dpd','pmt_n_90','pmt_ever_90plus','acc_n_with_overdue',
              'acc_open_ratio','acc_n_accounts','enq_n','enq_n_90d',
              'enq_days_since_last','is_cash_loan','has_accounts']
print(train.groupby('TARGET')[check_cols].mean().T)

corr = train[feature_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
print('\nTOP + correlated with default:')
print(corr.sort_values(ascending=False).head(15))
print('\nTOP - correlated with default:')
print(corr.sort_values().head(10))

TARGET                        0           1
pmt_max_dpd           29.938950   26.672508
pmt_n_90               0.562765    0.495037
pmt_ever_90plus        0.096644    0.084010
acc_n_with_overdue     0.010477    0.027212
acc_open_ratio         0.350902    0.407600
acc_n_accounts         4.775963    4.631334
enq_n                  7.303047    7.352139
enq_n_90d              1.703902    1.464691
enq_days_since_last  114.046657  134.840956
is_cash_loan           0.902204    0.935603
has_accounts           0.859919    0.819537

TOP + correlated with default:
enq_days_since_first         0.071712
acc_open_ratio               0.048146
enq_span_days                0.047705
acc_n_open                   0.044663
acc_cnt_Microloan_share      0.039542
acc_n_with_overdue           0.037761
enq_days_since_last          0.032345
is_cash_loan                 0.030986
acc_cnt_Microloan            0.029550
acc_overdue_ratio            0.027708
acc_cnt_Credit card          0.021046
acc_cnt_Credit card_sh

In [12]:
# Save
train.to_csv(OUT_TRAIN, index=False)
test.to_csv(OUT_TEST, index=False)
print(f'Saved {OUT_TRAIN} ({train.shape}) and {OUT_TEST} ({test.shape})')

Saved features_train.csv ((261383, 72)) and features_test.csv ((46127, 70))
